In [2]:
import json
import pandas as pd
import os

def parse_im3_offers(json_filepath):
    print(f"[*] Membuka berkas data mentah: {json_filepath}")
    with open(json_filepath, 'r') as file:
        data = json.load(file)

    # Validasi struktur respons API IOH
    if data.get('status') != "0" or 'commercial_package' not in data.get('data', {}):
        raise ValueError("Format JSON tidak valid atau status API mengembalikan error.")

    packages = data['data']['commercial_package']
    print(f"[*] Terdeteksi {len(packages)} paket komersial di dalam berkas JSON.")
    
    extracted_data = []

    for pkg in packages:
        commercial_attr = pkg.get('commercial_attribute', {})
        
        # Konversi kuota dari MB ke GB murni (skalar)
        try:
            total_quota_mb = float(commercial_attr.get('total_data_quota', 0))
            quota_gb = round(total_quota_mb / 1024, 2) if total_quota_mb > 0 else 0.0
        except (ValueError, TypeError):
            quota_gb = 0.0

        # Penanganan tipe data harga (skalar integer)
        try:
            price = int(pkg.get('tariff', 0))
            original_price = int(pkg.get('original_tariff', 0))
        except (ValueError, TypeError):
            price = 0
            original_price = 0
        
        extracted_data.append({
            'offer_id': pkg.get('pvr_code'),
            'package_name': pkg.get('package_name'),
            'family_name': commercial_attr.get('family_name'),
            'quota_gb': quota_gb,
            'duration_days': commercial_attr.get('duration_month'),
            'price_rp': price,
            'original_price_rp': original_price,
            'marketing_ribbon': pkg.get('package_ribbon'),
            'vas_included': pkg.get('package_footer_desc'),
            'auto_renew': commercial_attr.get('auto_renew')
        })

    df = pd.DataFrame(extracted_data)
    
    # Rekayasa Fitur: Rasio efisiensi harga per Gigabyte
    # Ditambahkan konstanta kecil 0.01 untuk mencegah error pembagian dengan nol (ZeroDivisionError) jika kuota 0
    df['price_per_gb'] = round(df['price_rp'] / (df['quota_gb'].apply(lambda x: x if x > 0 else 0.01)), 2)
    
    return df

if __name__ == "__main__":
    # Mendefinisikan jalur berkas secara absolut dari direktori kerja saat ini
    base_dir = "dataset_indosat"
    json_file = os.path.join(base_dir, "im3_offers_raw.json")
    csv_output = os.path.join(base_dir, "im3_pricing_architecture.csv")

    print("[*] Menginisialisasi Pipeline Ekstraksi Data Katalog Indosat...")
    
    # Pengecekan keberadaan file dengan pesan log balik yang eksplisit
    if not os.path.exists(json_file):
        print(f"\n[CRITICAL ERROR] Berkas '{json_file}' TIDAK DITEMUKAN!")
        print(f"-> Pastikan kamu sudah membuat folder '{base_dir}' di direktori kerjamu.")
        print(f"-> Pastikan salinan JSON respons tadi disimpan dengan nama 'im3_offers_raw.json' di dalam folder tersebut.\n")
    else:
        try:
            df_pricing = parse_im3_offers(json_file)
            
            # Ekspor ke CSV
            df_pricing.to_csv(csv_output, index=False)
            print(f"\n[SUCCESS] Pipeline berhasil dieksekusi.")
            print(f"-> Total Baris Data Terproses: {len(df_pricing)}")
            print(f"-> Berkas CSV disimpan di        : {csv_output}\n")
            
            # Tampilkan 5 baris pertama di terminal sebagai sampel validasi
            print("=== Sampel Matriks Data Hasil Parsing ===")
            print(df_pricing[['package_name', 'quota_gb', 'price_rp', 'price_per_gb']].head().to_string())
            
        except Exception as e:
            print(f"\n[EXECUTION FAILED] Terjadi kegagalan saat parsing data: {e}\n")

[*] Menginisialisasi Pipeline Ekstraksi Data Katalog Indosat...

[CRITICAL ERROR] Berkas 'dataset_indosat/im3_offers_raw.json' TIDAK DITEMUKAN!
-> Pastikan kamu sudah membuat folder 'dataset_indosat' di direktori kerjamu.
-> Pastikan salinan JSON respons tadi disimpan dengan nama 'im3_offers_raw.json' di dalam folder tersebut.

